# bob, explained - Episode 8: Synthesis with yosys

Synthesis with yosys

Run `!pip install manim` and `from manim import *` once first, then this cell. Start at `-ql`; the class names listed at the top of the cell render a single section.


In [ ]:
%%manim -qm Ep08Yosys
# =============================================================================
#  bob, explained - EPISODE 8: Synthesis with yosys
#  Synthesis with yosys
#
#  GENERATED by docs/manim/build.py from docs/manim/parts/. Do not edit here.
#
#  Prerequisite (once per notebook, in a cell of its own):
#      !pip install manim
#      from manim import *
#
#  Quality on the magic line above:  -ql draft   -qm medium   -qh 1080p60
#
#  Render one section instead of the whole episode by putting any of these
#  class names on the magic line:
#      E08S1What
#      E08S2Passes
#      E08S3Proof
#      E08S4Stimulus
#      E08S5Files
# =============================================================================

# =============================================================================
#  shared prelude - palette, helpers and the BobScene base class.
#  docs/manim/build.py pastes this into the top of every episode cell.
# =============================================================================

from manim import *
import numpy as np

# ---------------------------------------------------------------- palette ----
BG    = "#11121a"
INK   = "#e8e8ea"
DIM   = "#8b93a7"
C_PY  = "#7aa2f7"   # blue    - Python / tools / the device description
C_VPR = "#f7768e"   # red     - VPR / external tools
C_RTL = "#9ece6a"   # green   - hardware, Verilog, things on the die
C_BIT = "#e0af68"   # amber   - configuration bits, FASM, the bitstream
C_GRF = "#bb9af7"   # purple  - graphs, JTAG, protocol
C_ERR = "#ff7a93"   # pink    - bugs, refusals, errors
MONO  = "monospace"


# ---------------------------------------------------------------- helpers ----
def mono(s, size=22, color=INK):
    """One line of monospace text (Pango crashes on '', so blanks become ' ')."""
    return Text(s if s else " ", font=MONO, font_size=size, color=color)


def code_block(lines, size=20, color=INK):
    g = VGroup(*[mono(l, size, color) for l in lines])
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.14)
    return g


def panel(mob, color=DIM, pad=0.32, fill=0.06):
    r = SurroundingRectangle(mob, color=color, buff=pad)
    r.set_fill(color, opacity=fill)
    return VGroup(r, mob)


def chip(label, color, w=2.6, h=0.95, size=22, weight="NORMAL"):
    box = RoundedRectangle(width=w, height=h, corner_radius=0.14,
                           color=color, stroke_width=3)
    box.set_fill(color, opacity=0.12)
    txt = Text(label, font_size=size, color=INK, weight=weight, line_spacing=0.75)
    if txt.width > w - 0.3:
        txt.scale_to_fit_width(w - 0.3)
    if txt.height > h - 0.2:
        txt.scale_to_fit_height(h - 0.2)
    return VGroup(box, txt.move_to(box.get_center()))


def arrow(a, b, color=DIM, buff=0.15, sw=3):
    return Arrow(a, b, buff=buff, color=color, stroke_width=sw,
                 max_tip_length_to_length_ratio=0.18)


def mux_symbol(color=C_RTL, h=1.9, w=0.8):
    """Classic trapezoid multiplexer symbol."""
    p = Polygon([-w / 2,  h / 2, 0], [w / 2,  h / 2 - 0.3, 0],
                [ w / 2, -h / 2 + 0.3, 0], [-w / 2, -h / 2, 0],
                color=color, stroke_width=3)
    p.set_fill(color, opacity=0.14)
    return p


def bitcells(n, size=0.3, on=(), color=C_BIT, off_color=DIM):
    """A strip of n little squares; indices in `on` are filled."""
    g = VGroup()
    for i in range(n):
        s = Square(size, color=off_color, stroke_width=1.6)
        if i in on:
            s.set_stroke(color).set_fill(color, opacity=0.85)
        g.add(s)
    g.arrange(RIGHT, buff=0.035)
    return g


def fieldbar(fields, total_w=11.0, h=0.62, size=15):
    """
    fields: [(label, nbits, color), ...] -> one horizontal bar split to scale,
    each slice labelled above and its bit range below. Returns VGroup(bar, labels, ranges).
    """
    nbits = sum(f[1] for f in fields)
    bar, labs, rngs = VGroup(), VGroup(), VGroup()
    x, lo = -total_w / 2, 0
    for label, n, col in fields:
        w = max(total_w * n / nbits, 0.34)
        r = Rectangle(width=w, height=h, color=col, stroke_width=2)
        r.set_fill(col, opacity=0.28).move_to(np.array([x + w / 2, 0, 0]))
        bar.add(r)
        t = Text(label, font_size=size, color=col)
        if t.width > w * 1.9:
            t.scale_to_fit_width(max(w * 1.9, 0.5))
        t.next_to(r, UP, buff=0.14)
        labs.add(t)
        rt = mono(f"{lo}" if n == 1 else f"{lo}..{lo + n - 1}", size - 2, DIM)
        rt.next_to(r, DOWN, buff=0.12)
        if rt.width > w * 1.9:
            rt.scale_to_fit_width(max(w * 1.9, 0.5))
        rngs.add(rt)
        x += w
        lo += n
    return VGroup(bar, labs, rngs)


def filecard(path, role, color):
    """A small card naming a repo file and what it is."""
    t = mono(path, 17, color)
    r = Text(role, font_size=14, color=DIM)
    g = VGroup(t, r).arrange(DOWN, aligned_edge=LEFT, buff=0.08)
    box = SurroundingRectangle(g, color=color, buff=0.16)
    box.set_fill(color, opacity=0.07)
    return VGroup(box, g)


def mid(a, b):
    """midpoint, defined here so nothing depends on manim exporting space_ops."""
    return (a + b) / 2


def clear_all(sc, run_time=0.6):
    if sc.mobjects:
        sc.play(*[FadeOut(m) for m in sc.mobjects], run_time=run_time)


class BobScene(Scene):
    def setup(self):
        self.camera.background_color = BG

    def heading(self, text, kicker=None):
        t = Text(text, font_size=32, color=INK, weight="BOLD")
        t.to_corner(UL).shift(DOWN * 0.1)
        rule = Line(LEFT * 6.6, RIGHT * 6.6, color=DIM, stroke_width=1.5)
        rule.next_to(t, DOWN, buff=0.2).align_to(t, LEFT)
        g = VGroup(t, rule)
        self.play(FadeIn(t, shift=RIGHT * 0.3), Create(rule), run_time=0.7)
        if kicker:
            k = Text(kicker, font_size=19, color=DIM)
            if k.width > 13.0:
                k.scale_to_fit_width(13.0)
            k.next_to(rule, DOWN, buff=0.16).align_to(t, LEFT)
            g.add(k)
            self.play(FadeIn(k), run_time=0.4)
        return g

    def titlecard(self, number, title, subtitle):
        n = Text(number, font_size=26, color=C_BIT, weight="BOLD")
        t = Text(title, font_size=60, color=INK, weight="BOLD")
        s = Text(subtitle, font_size=26, color=DIM)
        if t.width > 12.5:
            t.scale_to_fit_width(12.5)
        if s.width > 12.5:
            s.scale_to_fit_width(12.5)
        g = VGroup(n, t, s).arrange(DOWN, buff=0.4)
        self.play(FadeIn(n), run_time=0.4)
        self.play(Write(t), run_time=1.1)
        self.play(FadeIn(s, shift=UP * 0.2), run_time=0.7)
        self.wait(1.6)
        self.play(FadeOut(g), run_time=0.6)

    def files_used(self, inputs, generated, verified):
        """Closing card: what this episode's topic is built from and checked by."""
        self.heading("Files", "what this part is written in, what is generated, and what proves it")
        cols = []
        for title, items, col in (("written by hand", inputs, C_RTL),
                                  ("generated", generated, C_PY),
                                  ("verified by", verified, C_BIT)):
            head = Text(title, font_size=21, color=col, weight="BOLD")
            cards = VGroup(*[filecard(p, r, col) for p, r in items])
            cards.arrange(DOWN, aligned_edge=LEFT, buff=0.18)
            g = VGroup(head, cards).arrange(DOWN, aligned_edge=LEFT, buff=0.28)
            cols.append(g)
        row = VGroup(*cols).arrange(RIGHT, buff=0.7, aligned_edge=UP)
        if row.width > 13.2:
            row.scale_to_fit_width(13.2)
        row.next_to(self.mobjects[1], DOWN, buff=0.55).set_x(0)
        for c in cols:
            self.play(FadeIn(c, shift=UP * 0.2), run_time=0.7)
        self.wait(2.4)

# =============================================================================
#  EPISODE 8 - Synthesis: turning Verilog into cells bob actually has
# =============================================================================

def s1_what(sc):
    sc.heading("Synthesis is translation, not magic",
               "tools/bob/synth.py drives yosys with bob's own cell library and maps")

    src = chip("counter.v\nordinary Verilog", INK, 3.0, 1.2, 20).move_to(np.array([-4.6, 1.5, 0]))
    ys = chip("yosys", C_VPR, 2.2, 1.0, 24, "BOLD").move_to(np.array([-0.6, 1.5, 0]))
    out = chip("only bob cells", C_RTL, 3.2, 1.2, 20).move_to(np.array([3.6, 1.5, 0]))
    sc.play(FadeIn(src), run_time=0.4)
    sc.play(GrowArrow(arrow(src.get_right(), ys.get_left(), DIM, 0.1)), FadeIn(ys),
            run_time=0.5)
    sc.play(GrowArrow(arrow(ys.get_right(), out.get_left(), DIM, 0.1)), FadeIn(out),
            run_time=0.5)

    cells = code_block([
        "$lut         a K-input LUT             -> one CLB",
        "BOB_ADD      one carry bit             -> one CLB in carry mode",
        "BOB_FDRE     flip-flop, sync reset     -> the CLB's flop, ff_rstval = 0",
        "BOB_FDSE     flip-flop, sync set       -> ff_rstval = 1",
        "BOB_BRAM18   1024 x 18 true dual port  -> a BRAM block",
        "BOB_DSP      25 x 18 signed            -> a DSP slice",
    ], 19, C_RTL)
    cells.next_to(out, DOWN, buff=1.0).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l, shift=RIGHT * 0.15) for l in cells], lag_ratio=0.15),
            run_time=1.9)

    strict = Text("synth.py FAILS if anything else survives, or if the design has more "
                  "than one clock. There is no 'mostly mapped'.",
                  font_size=20, color=C_BIT)
    strict.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(strict), run_time=0.9)
    sc.wait(2.0)


def s2_passes(sc):
    sc.heading("The pass order", "patterned on yosys's own synth_xilinx, with bob's maps swapped in")

    steps = [
        ("read + hierarchy", "elaborate, pick the top", DIM),
        ("proc, flatten, opt", "ordinary front-end work", DIM),
        ("mul2dsp 25x18", "big multipliers become BOB_DSP", C_BIT),
        ("memory_libmap", "memories become BOB_BRAM18, rules in bob_brams.txt", C_BIT),
        ("techmap _80_bob_alu", "$alu becomes BOB_ADD carry chains", C_RTL),
        ("dfflegalize", "every flop becomes BOB_FDRE / BOB_FDSE", C_RTL),
        ("abc -lut K", "whatever is left becomes $lut", C_VPR),
        ("write json / blif / sim", "for the placer, for VPR, for simulation", C_PY),
    ]
    g = VGroup()
    for a, b, col in steps:
        g.add(VGroup(mono(a, 20, col), Text(b, font_size=17, color=DIM))
              .arrange(RIGHT, buff=0.5, aligned_edge=DOWN))
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 4.6)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.26)
    if g.width > 13.0:
        g.scale_to_fit_width(13.0)
    g.next_to(sc.mobjects[1], DOWN, buff=0.6).set_x(0)
    for r in g:
        sc.play(FadeIn(r, shift=RIGHT * 0.2), run_time=0.35)

    trap = code_block([
        "A trap worth remembering: the rule was first called _90_bob_alu, and yosys",
        "picked the generic _90_alu instead - rules are tried in NAME order.",
        "Renaming it _80_ fixed it. Nothing errored; the carry chains just vanished.",
    ], 18, C_ERR)
    trap.to_edge(DOWN, buff=0.35).set_x(0)
    sc.play(FadeIn(trap), run_time=0.9)
    sc.wait(2.2)


def s3_proof(sc):
    sc.heading("Three copies of the same circuit, simulated together",
               "tools/bob/equiv.py - this is what stops a wrong map from ever reaching the board")

    three = VGroup(
        chip("the source\ncounter.v", INK, 3.0, 1.2, 19),
        chip("the yosys netlist\ncounter_syn.v", C_VPR, 3.4, 1.2, 19),
        chip("the golden netlist\ngolden.py, one wire per bit", C_PY, 4.2, 1.2, 17),
    ).arrange(RIGHT, buff=0.6).shift(UP * 1.8)
    sc.play(LaggedStart(*[FadeIn(t) for t in three], lag_ratio=0.2), run_time=1.0)

    sim = chip("iverilog: all three, same stimulus, 300 cycles", C_BIT, 8.6, 0.9, 20)
    sim.next_to(three, DOWN, buff=0.8)
    for t in three:
        sc.play(GrowArrow(arrow(t.get_bottom(), sim.get_top(), DIM, 0.1)), run_time=0.2)
    sc.play(FadeIn(sim), run_time=0.5)

    chk = code_block([
        "compared before AND after every clock edge",
        "the source trace is saved   -> later checked against model.py and the board",
        "every golden net is saved   -> later checked against CAPTURE on the board",
    ], 19, INK)
    chk.next_to(sim, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in chk], lag_ratio=0.2), run_time=1.5)

    why = Text("Why a third, 'golden' netlist? Because yosys's own output renames every "
               "cell and net, so nothing in it maps back to a fabric location. "
               "golden.py writes the same logic with one named wire per bit.",
               font_size=18, color=DIM)
    why.scale_to_fit_width(13.0).next_to(chk, DOWN, buff=0.5)
    sc.play(FadeIn(why), run_time=0.9)
    sc.wait(2.2)


def s4_stimulus(sc):
    sc.heading("The bug that made this episode necessary",
               "a test that passes for the wrong reason is worse than no test")

    story = code_block([
        "M8: the counter example passed. Model, RTL and board all agreed.",
        "",
        "M9: someone looked at the saved trace. It was all zeros.",
    ], 22, INK)
    story[2].set_color(C_ERR)
    story.shift(UP * 1.9)
    sc.play(FadeIn(story[0]), run_time=0.7)
    sc.wait(0.6)
    sc.play(FadeIn(story[2]), run_time=0.7)
    sc.wait(1.0)

    why = code_block([
        "The stimulus was uniform random over all inputs.",
        "The counter's synchronous reset was one of those inputs.",
        "So it was asserted about half the time - and the counter never counted.",
        "",
        "Every check compared zero against zero and passed.",
    ], 20, INK)
    why[4].set_color(C_ERR)
    why.next_to(story, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in why], lag_ratio=0.2), run_time=1.9)

    fix = code_block([
        "The fix: biased vectors in 50-cycle segments, so control inputs hold still",
        "long enough for state to build up. The counter's trace now reaches LED 0-6.",
        "",
        "The rule: check that your stimulus actually exercises what it claims.",
    ], 20, C_BIT)
    fix[3].set_color(C_RTL)
    fix.next_to(why, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in fix], lag_ratio=0.2), run_time=1.7)
    sc.wait(2.2)


def s5_files(sc):
    sc.files_used(
        inputs=[("tools/bob/synth/bob_cells_sim.v", "the cell library, simulatable"),
                ("tools/bob/synth/bob_map.v", "techmap rules, incl. _80_bob_alu"),
                ("tools/bob/synth/bob_brams.txt", "memory_libmap rules"),
                ("examples/*.v", "the designs themselves")],
        generated=[("build/synth/<top>/<top>.json", "for the placer and VPR"),
                   ("build/synth/<top>/<top>_syn.v", "the netlist, for simulation"),
                   ("tools/bob/golden.py output", "one named wire per bit"),
                   ("trace.json", "the source trace + every golden net per clock")],
        verified=[("tools/bob/equiv.py", "source == netlist == golden, 300 cycles"),
                  ("hw/tb/tb_synth.v", "972 checks: the designs on the real fabric"),
                  ("tests/test_synth.py", "non-bob cells rejected, chains split by column")])


EP08 = [s1_what, s2_passes, s3_proof, s4_stimulus, s5_files]


class Ep08Yosys(BobScene):
    def construct(self):
        self.titlecard("EPISODE 8", "Synthesis",
                       "Verilog into cells bob actually has")
        for i, part in enumerate(EP08):
            part(self)
            if i < len(EP08) - 1:
                clear_all(self)


class E08S1What(BobScene):
    def construct(self): s1_what(self)


class E08S2Passes(BobScene):
    def construct(self): s2_passes(self)


class E08S3Proof(BobScene):
    def construct(self): s3_proof(self)


class E08S4Stimulus(BobScene):
    def construct(self): s4_stimulus(self)


class E08S5Files(BobScene):
    def construct(self): s5_files(self)